# 04. Customer, Regional and Discount Analysis

Objective: Analyze customer contribution, regional performance, and the relationship between discounts and profit, and practice NumPy.
Input: data/processed/cleaned_sales_data.csv

## 1. Project Setup

### 1.1 Import Libraries

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

### 1.2 Environment Check

In [2]:
print("Numpy version: ", np.__version__)
print("Pandas version: ", pd.__version__)

Numpy version:  2.5.4
Pandas version:  3.1.1


### 1.3 Set Project Path

In [3]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_sales_data.csv"

print("Project Root: ", PROJECT_ROOT)
print("Dataset exists: ", DATA_PATH.exists())
print("Dataset path: ", DATA_PATH)

Project Root:  c:\Users\pc\Desktop\Retail-Sales-and-Business-Analysis
Dataset exists:  True
Dataset path:  c:\Users\pc\Desktop\Retail-Sales-and-Business-Analysis\data\processed\cleaned_sales_data.csv


### 1.4 Load Cleaned Dataset

Objective: Load the cleaned dataset from Day 2 with the date columns parsed as dates.

In [4]:
sales_df = pd.read_csv(
    DATA_PATH,
    parse_dates=['Order Date', 'Ship Date']
)
print(sales_df[['Order Date', 'Ship Date']].dtypes)
print("Dataset shape:", sales_df.shape)

Order Date    datetime64[us]
Ship Date     datetime64[us]
dtype: object
Dataset shape: (10194, 26)


### 1.5 Verify Dataset Structure

Objective: Verify that the dataset loaded correctly, with the expected shape and data types.


In [5]:
print(sales_df.dtypes)
print(sales_df.head(3))

Row ID                     int64
Order ID                     str
Order Date        datetime64[us]
Ship Date         datetime64[us]
Ship Mode                    str
Customer ID                  str
Customer Name                str
Segment                      str
Country/Region               str
City                         str
State/Province               str
Postal Code                  str
Region                       str
Product ID                   str
Category                     str
Sub-Category                 str
Product Name                 str
Sales                    float64
Quantity                   int64
Discount                 float64
Profit                   float64
Order Year                 int64
Order Month                int64
Year-Month                   str
Shipping Days              int64
Profit Status                str
dtype: object
   Row ID        Order ID Order Date  Ship Date       Ship Mode Customer ID  \
0       1  US-2023-103800 2023-01-03 2023-01-07  

Observations:

The cleaned dataset loaded successfully with 10,194 rows and 26 columns, matching the Day 2 output. Order Date and Ship Date were parsed as datetime columns, and the five engineered features (Order Year, Order Month, Year-Month, Shipping Days, Profit Status) are present.

In [6]:
total_sales = sales_df['Sales'].sum()
total_profit = sales_df['Profit'].sum()

## 2. Customer Analysis

Objective: Understand customer contribution to sales and profit, segment performance, and purchasing frequency.

### 2.1 Top Customers

Objective: Identify the ten customers with the highest total sales and the ten with the highest total profit.

#### 2.1.1 Top 10 Customers by Sales

In [7]:
top_customers_sales = sales_df.groupby(['Customer ID', 'Customer Name'])['Sales'].sum().sort_values(ascending=False).head(10).round(2)
print("Top 10 customers by sales:\n")
print(top_customers_sales)

Top 10 customers by sales:

Customer ID  Customer Name     
SM-20320     Sean Miller           25043.05
TC-20980     Tamara Chand          19052.22
RB-19360     Raymond Buch          15117.34
TA-21385     Tom Ashbrook          14595.62
AB-10105     Adrian Barton         14473.57
KL-16645     Ken Lonsdale          14175.23
SC-20095     Sanjit Chand          14142.33
HL-15040     Hunter Lopez          12873.30
SE-20110     Sanjit Engle          12209.44
CC-12370     Christopher Conant    12129.07
Name: Sales, dtype: float64


#### 2.1.2 Top 10 Customers by Profit

In [8]:
top_customers_profit = sales_df.groupby(['Customer ID', 'Customer Name'])['Profit'].sum().sort_values(ascending=False).head(10).round(2)
print("Top 10 customers by profit:\n")
print(top_customers_profit)

Top 10 customers by profit:

Customer ID  Customer Name       
TC-20980     Tamara Chand            8981.32
RB-19360     Raymond Buch            6976.10
SC-20095     Sanjit Chand            5757.41
HL-15040     Hunter Lopez            5622.43
AB-10105     Adrian Barton           5444.81
TA-21385     Tom Ashbrook            4703.79
CM-12385     Christopher Martinez    3899.89
KD-16495     Keith Dawkins           3038.63
AR-10540     Andy Reiter             2884.62
DR-12940     Daniel Raglin           2869.08
Name: Profit, dtype: float64


Observations:

- The customer with the highest total sales is Sean Miller (25,043.05), followed by Tamara Chand (19,052.22) and Raymond Buch (15,117.34).
- The customer with the highest total profit is Tamara Chand (8,981.32), followed by Raymond Buch (6,976.10) and Sanjit Chand (5,757.41).
- Six customers appear in both top 10 lists: Tamara Chand, Raymond Buch, Sanjit Chand, Hunter Lopez, Adrian Barton and Tom Ashbrook.
- Sean Miller is the top customer by sales but is not among the top 10 customers by profit, so the highest sales do not always mean the highest profit.

### 2.2 Customer Segment Performance

Objective: Compare sales, profit, and profit margin across customer segments.


#### 2.2.1 Sales and Profit by Segment

In [9]:
segment_summary = sales_df.groupby('Segment')[['Sales', 'Profit']].sum()
print(segment_summary)

                    Sales       Profit
Segment                               
Consumer     1.170660e+06  136371.4463
Corporate    7.158061e+05   94249.6400
Home Office  4.400684e+05   61675.7283


#### 2.2.2 Profit Margin by Segment

In [10]:
segment_margin = (segment_summary['Profit'] / segment_summary['Sales'] * 100).round(2)
print("Profit margin by segment (%):\n")
print(segment_margin)

print("\nSegment sales add upto total sales:", np.isclose(segment_summary['Sales'].sum(),  total_sales))
print("Segment profit add upto total profit:", np.isclose(segment_summary['Profit'].sum(),  total_profit))

Profit margin by segment (%):

Segment
Consumer       11.65
Corporate      13.17
Home Office    14.02
dtype: float64

Segment sales add upto total sales: True
Segment profit add upto total profit: True


Observations:

- Consumer is the largest segment, with sales of about 1,170,660 and profit of 136,371.45, followed by Corporate (sales about 715,806, profit 94,249.64) and Home Office (sales about 440,068, profit 61,675.73).
- Profit margins are 11.65% for Consumer, 13.17% for Corporate and 14.02% for Home Office. Home Office is the smallest segment by sales but has the highest margin.
- The three segments add up to the overall total sales and total profit.

### 2.3 Orders per Customer

Objective: Understand how often customers place orders.

#### 2.3.1 Orders per Customer Summary

In [11]:
orders_per_customer = sales_df.groupby(['Customer ID', 'Customer Name'])['Order ID'].nunique()
print(orders_per_customer)
print("Customers:", len(orders_per_customer))
print(orders_per_customer.describe())

Customer ID  Customer Name     
AA-10315     Alex Avila             5
AA-10375     Allen Armold           9
AA-10480     Andrew Allen           4
AA-10645     Anna Andreadi          6
AB-10015     Aaron Bergman          3
                                   ..
XP-21865     Xylona Preis          13
YC-21895     Yoseph Carroll         5
YS-21880     Yana Sorensen          8
ZC-21910     Zuschuss Carroll      13
ZD-21925     Zuschuss Donatelli     5
Name: Order ID, Length: 804, dtype: int64
Customers: 804
count    804.000000
mean       6.364428
std        2.660121
min        1.000000
25%        5.000000
50%        6.000000
75%        8.000000
max       17.000000
Name: Order ID, dtype: float64


#### 2.3.2 Repeat Customers

In [12]:
repeat_customers = orders_per_customer[orders_per_customer > 1]
repeat_pct = len(repeat_customers) / len(orders_per_customer) * 100

print("\nRepeat customers:", len(repeat_customers))
print(f"Repeat customers share: {repeat_pct:.2f}%")


Repeat customers: 792
Repeat customers share: 98.51%


Observations:

- The dataset has 804 customers, who placed between 1 and 17 orders each. The average is about 6.4 orders per customer and the median is 6.
- 792 customers (98.51%) placed more than one order, so only 12 customers ordered once.
- Almost all customers in this dataset are repeat customers. This includes the future-dated records retained in Day 2.

## 3. Regional Analysis

Objective: Compare sales and profitability across regions and states.

### 3.1 Region Performance

Objective: Calculate total sales, total profit, and profit margin for each region.


#### 3.1.1 Sales and Profit by Region

In [13]:
region_summary = sales_df.groupby('Region')[['Sales', 'Profit']].sum().sort_values('Sales', ascending=False)
print(region_summary.round(2))


             Sales     Profit
Region                       
West     739813.61  110798.82
East     691828.17   94883.26
Central  503170.67   39865.31
South    391721.90   46749.43


#### 3.1.2 Profit Margin by Region

In [14]:
region_margin = (region_summary['Profit'] / region_summary['Sales'] * 100).round(2)
print("Profit margin by region (%):\n")
print(region_margin)

print("Region sales add up to total sales:", np.isclose(region_summary['Sales'].sum(), total_sales))
print("Region profit add up to total profit:", np.isclose(region_summary['Profit'].sum(), total_profit))
print("Missing Region values:", sales_df['Region'].isnull().sum())

Profit margin by region (%):

Region
West       14.98
East       13.71
Central     7.92
South      11.93
dtype: float64
Region sales add up to total sales: True
Region profit add up to total profit: True
Missing Region values: 0


Observations:

- Profit margins by region are 14.98% for West, 13.71% for East, 11.93% for South and 7.92% for Central.
- West has the highest margin and Central has the lowest, well below the other three regions.
- The four regions add up to the overall total sales and total profit.

### 3.2 State-Wise Performance

Objective: Identify the states with the highest and lowest profit.

#### 3.2.1 Top 10 States by Profit

In [15]:
state_profit = sales_df.groupby('State/Province')['Profit'].sum().sort_values(ascending=False).round(2)

top_states = state_profit.head(10)

print("Top 10 states by profit:")
print(top_states)

print("\nMissing state values:", sales_df['State/Province'].isnull().sum())

Top 10 states by profit:
State/Province
California    76381.39
New York      74038.55
Washington    33402.65
Michigan      24463.19
Virginia      18597.95
Indiana       18382.94
Georgia       16250.04
Kentucky      11199.70
Minnesota     10823.19
Delaware       9977.37
Name: Profit, dtype: float64

Missing state values: 0


#### 3.2.2 Bottom 10 States by Profit

In [16]:
bottom_states = state_profit.tail(10)

print("\nBottom 10 states by profit:")
print(bottom_states)


Bottom 10 states by profit:
State/Province
Oregon            -1190.47
Florida           -3399.30
Arizona           -3427.92
Tennessee         -5341.69
Colorado          -6527.86
North Carolina    -7490.91
Illinois         -12607.89
Pennsylvania     -15559.96
Ohio             -16971.38
Texas            -25729.36
Name: Profit, dtype: float64


Observations:

- California (76,381.39) and New York (74,038.55) have the highest profit, followed by Washington (33,402.65).
- All of the 10 lowest states by profit are loss-making. Texas has the largest loss (-25,729.36), followed by Ohio (-16,971.38) and Pennsylvania (-15,559.96).
- Profit is concentrated: California and New York together earn far more than any other state.

### 3.3 Best and Worst Performing Regions

In [17]:
print("Highest sales region:", region_summary['Sales'].idxmax())
print("Lowest sales region:", region_summary['Sales'].idxmin())

print("\nHighest profit region:", region_summary['Profit'].idxmax())
print("Lowest profit region:", region_summary['Profit'].idxmin())

Highest sales region: West
Lowest sales region: South

Highest profit region: West
Lowest profit region: Central



Observations:

- West has the highest sales (739,813.61) and highest profit (110,798.82), making it the strongest region overall.
- South has the lowest sales (391,721.90), while Central has the lowest profit (39,865.31).
- West has the highest profit margin (14.98%), indicating the strongest profitability relative to sales.
- Central has the lowest profit margin (7.92%), despite generating more sales than South.
- The results show that higher sales do not necessarily translate into higher profitability.

## 4. Discount Analysis

Objective: Investigate how discounts relate to profit and whether they help explain loss-making records.


### 4.1 Discount Distribution


#### 4.1.1 Records by Discount Level

In [18]:
discount_counts = sales_df['Discount'].value_counts().sort_index()
discount_share = (sales_df['Discount'].value_counts(normalize=True).sort_index()*100).round(1)
print(discount_counts)
print("\nShare of records (%):")
print(discount_share)

Discount
0.00    4925
0.10      96
0.15      52
0.20    3706
0.30     230
0.32      27
0.40     207
0.45      11
0.50      66
0.60     149
0.70     424
0.80     301
Name: count, dtype: int64

Share of records (%):
Discount
0.00    48.3
0.10     0.9
0.15     0.5
0.20    36.4
0.30     2.3
0.32     0.3
0.40     2.0
0.45     0.1
0.50     0.6
0.60     1.5
0.70     4.2
0.80     3.0
Name: proportion, dtype: float64


Observations:

- There are 12 discount levels. No discount (0.00) is the most common with 4,925 records (48.3%), followed by 0.20 with 3,706 records (36.4%). Together they cover about 84.7% of records.
- The higher levels are less frequent: 0.70 has 424 records (4.2%) and 0.80 has 301 records (3.0%).

### 4.2 Profit Across Discount Levels

#### 4.2.1 Sales, Profit and Margin by Discount Level

In [19]:
discount_summary = sales_df.groupby('Discount')[['Sales', 'Profit']].sum()
discount_summary['Records'] = sales_df['Discount'].value_counts()
discount_summary['Margin %'] = (discount_summary['Profit'] / discount_summary['Sales'] * 100)

print(discount_summary.round(2))

print("\nSales add up to total:", np.isclose(discount_summary['Sales'].sum(), total_sales))
print("Profit adds up to total:", np.isclose(discount_summary['Profit'].sum(), total_profit))

               Sales     Profit  Records  Margin %
Discount                                          
0.00      1105323.79  326718.59     4925     29.56
0.10        54952.50    9099.97       96     16.56
0.15        27558.52    1418.99       52      5.15
0.20       773939.40   91079.95     3706     11.77
0.30       104474.07  -10513.45      230    -10.06
0.32        14493.46   -2391.14       27    -16.50
0.40       116497.76  -23086.37      207    -19.82
0.45         5484.97   -2493.11       11    -45.45
0.50        58918.54  -20506.43       66    -34.80
0.60         7105.94   -6164.35      149    -86.75
0.70        40805.27  -40300.57      424    -98.76
0.80        16980.15  -30565.27      301   -180.01

Sales add up to total: True
Profit adds up to total: True


Observations:

- Records with no discount earn a margin of 29.56% (profit 326,718.59 on sales 1,105,323.79).
- The margin is 16.56% at 0.10, 5.15% at 0.15 and 11.77% at 0.20.
- From 0.30 upward, profit is negative at every level, and the margin reaches -86.75% at 0.60, -98.76% at 0.70 and -180.01% at 0.80.
- The discount levels add up to the overall total sales and total profit.

### 4.3 Discount vs Profit Relationship

#### 4.3.1 Discounted vs Non-Discounted Records

In [20]:
has_discount = sales_df['Discount'] > 0
print(sales_df.groupby(has_discount)['Profit'].agg(['count', 'sum', 'mean']).round(2))

          count        sum   mean
Discount                         
False      4925  326718.59  66.34
True       5269  -34421.77  -6.53


#### 4.3.2 Correlation Between Discount and Profit

In [21]:
print("Correlation (Discount vs Profit):", round((sales_df['Discount'].corr(sales_df['Profit'])), 3))

Correlation (Discount vs Profit): -0.219


Observations:

- The 4,925 non-discounted records earn a total profit of 326,718.59 (average 66.34), while the 5,269 discounted records have a total profit of -34,421.77 (average -6.53).
- The correlation between Discount and Profit is -0.219, a weak to moderate negative relationship. Higher discounts are associated with lower profit.
- This is an association, not proof of cause: other factors such as product mix and costs are not in this dataset.

### 4.4 Loss-Making Records by Discount Level

In [22]:
# 4.4 Loss-Making Records by Discount Level
loss_rate = ((sales_df['Profit'] < 0).groupby(sales_df['Discount']).mean() * 100).round(1)
print("Share of loss-making records by discount level (%):")
print(loss_rate)

Share of loss-making records by discount level (%):
Discount
0.00      0.0
0.10      4.2
0.15     32.7
0.20     13.8
0.30     91.7
0.32    100.0
0.40     87.4
0.45    100.0
0.50    100.0
0.60    100.0
0.70    100.0
0.80    100.0
Name: Profit, dtype: float64


Observations:

- No record at 0.00 discount is loss-making. The share rises to 13.8% at 0.20, 91.7% at 0.30 and 87.4% at 0.40.
- Every record at 0.32, 0.45, 0.50, 0.60, 0.70 and 0.80 is loss-making (100%).

### 4.5 Average Discount by Sub-Category

In [23]:
subcategory_discount = sales_df.groupby('Sub-Category')['Discount'].mean().sort_values(ascending=False).round(3)
print(subcategory_discount)

Sub-Category
Binders        0.369
Machines       0.304
Tables         0.258
Bookcases      0.215
Chairs         0.169
Appliances     0.165
Copiers        0.157
Phones         0.153
Furnishings    0.138
Envelopes      0.080
Fasteners      0.079
Accessories    0.078
Supplies       0.076
Storage        0.075
Art            0.075
Paper          0.075
Labels         0.071
Name: Discount, dtype: float64


Observations:

- Binders (0.369), Machines (0.304), Tables (0.258) and Bookcases (0.215) have the highest average discounts.
- Machines, Tables and Bookcases were also weak or loss-making in Day 3, which is consistent with high discounts, but this does not prove the cause.
- Supplies is loss-making but has a low average discount (0.076), so discounts alone do not explain its loss. Binders has the highest average discount yet is profitable overall.

Key Findings:

- 84.7% of records have either no discount or a 20% discount.
- No-discount records earn a 29.56% margin, while every discount level from 30% upward is loss-making in total.
- Discounted records are loss-making overall (-34,421.77), and the Discount-Profit correlation is -0.219.
- Tables, Bookcases and Machines have high average discounts, but Supplies and Binders show that discounts do not explain every result.

Decision / Conclusion:

High discounts are strongly associated with losses in this data, but the analysis cannot prove they cause them. Pricing, product costs and product mix are not available here.

## 5. NumPy Practice

Objective: Practice NumPy array operations using columns from the retail dataset.

### 5.1 Arrays and Data Types

In [24]:
sales_arr = sales_df['Sales'].to_numpy()
profit_arr = sales_df['Profit'].to_numpy()
quantity_arr = sales_df['Quantity'].to_numpy()

print(type(sales_arr), sales_arr.dtype, sales_arr.shape)

<class 'numpy.ndarray'> float64 (10194,)


### 5.2 Indexing and Slicing

In [25]:
print(sales_arr[0])
print(sales_arr[:5])
print(sales_arr[-5:])

16.448
[ 16.448   3.54   11.784 272.736  19.536]
[52.776 20.72   3.024 90.93   3.024]


### 5.3 Reshape and Dimensions

In [26]:
matrix = sales_arr[:12].reshape(3,4)
print(matrix)
print(matrix.shape, matrix.ndim)

[[  16.448    3.54    11.784  272.736]
 [  19.536 2573.82     5.48    12.78 ]
 [ 609.98    31.12     6.54    19.44 ]]
(3, 4) 2


### 5.4 Vectorized Operations

In [27]:
sales_per_unit = sales_arr / quantity_arr
cost_arr = sales_arr - profit_arr
print(sales_per_unit[:5].round(2))
print(cost_arr[:5].round(2))

[ 8.22  1.77  3.93 90.91  6.51]
[ 10.9    9.03   7.51 337.51  14.65]


### 5.5 Broadcasting

In [28]:
weights = np.array([1, 2, 3, 4])
print(matrix * weights)
print(matrix - matrix.mean(axis=0))

[[  16.448    7.08    35.352 1090.944]
 [  19.536 5147.64    16.44    51.12 ]
 [ 609.98    62.24    19.62    77.76 ]]
[[-1.98873333e+02 -8.65953333e+02  3.84933333e+00  1.71084000e+02]
 [-1.95785333e+02  1.70432667e+03 -2.45466667e+00 -8.88720000e+01]
 [ 3.94658667e+02 -8.38373333e+02 -1.39466667e+00 -8.22120000e+01]]


### 5.6 Boolean Masking

In [29]:
loss_mask = profit_arr < 0
print("Loss-making records:", loss_mask.sum())
print("Sales of loss-making records:", sales_arr[loss_mask].sum().round(2))

Loss-making records: 1901
Sales of loss-making records: 472248.76


### 5.7 Axis-Based Aggregation

In [30]:
print("Columns sums (axis=0):", matrix.sum(axis=0).round(2))
print("Rows sums (axis=1):", matrix.sum(axis=1).round(2))

Columns sums (axis=0): [ 645.96 2608.48   23.8   304.96]
Rows sums (axis=1): [ 304.51 2611.62  667.08]


### 5.8 Handling NaN Values

In [31]:
values = np.array([10.0, np.nan, 20.0])
print(np.isnan(values))
print(np.mean(values), np.nanmean(values))

[False  True False]
nan 15.0


### 5.9 Conditional Selection with where() and select()

In [32]:
status_where = np.where(profit_arr > 0, 'Profitable', 'Not Profitable')
status_select = np.select(
    [profit_arr > 0, profit_arr < 0, profit_arr == 0],
    ['Profitable', 'Loss', 'Break-even'],
    default='Unknown'
)
print(np.unique(status_select, return_counts=True))
print("Matches Profit Status column:", (status_select == sales_df['Profit Status'].to_numpy()).all())

(array(['Break-even', 'Loss', 'Profitable'], dtype='<U10'), array([  65, 1901, 8228]))
Matches Profit Status column: True


### 5.10 Statistics

In [33]:
print("Mean:", sales_arr.mean().round(2))
print("Median:", np.median(sales_arr).round(2))
print("Std (ddof=1):", sales_arr.std(ddof=1).round(2))
print("Percentiles 25/50/75:", np.percentile(sales_arr, [25, 50, 75]).round(2))

Mean: 228.23
Median: 53.91
Std (ddof=1): 619.91
Percentiles 25/50/75: [ 17.22  53.91 209.5 ]


### 5.11 NumPy Practice Summary

Key Findings:

- to_numpy() converts a column to an array; shape and dtype describe it.
- Slicing and indexing select elements by position; reshape changes the dimensions.
- Vectorized operations work on whole arrays without loops; broadcasting combines arrays of compatible shapes.
- Boolean masks select elements by condition (1,901 loss-making records matched Day 1).
- axis=0 aggregates down columns and axis=1 across rows.
- np.isnan and np.nanmean handle missing values; where() and select() assign values by condition.
- Mean, median and standard deviation matched the Day 1 pandas results (std needs ddof=1 to match pandas).

## 6. Day 4 Summary

- Regions: West has the highest profit margin (14.98%) and Central the lowest (7.92%). California and New York have the highest profit among states, and Texas, Ohio and Pennsylvania have the largest losses.
- Discounts: 84.7% of records have no discount or a 20% discount. No-discount records earn a 29.56% margin, every discount level from 30% upward is loss-making in total, and the Discount-Profit correlation is -0.219.